# Dados e primeira análise com LLM

## 📌 Contexto e Objetivos
Este notebook consolida a auditoria e o saneamento preliminar dos dados transacionais legados no contexto de **PLD/AML (Prevenção à Lavagem de Dinheiro e Financiamento do Terrorismo)**.

### Fluxo de Trabalho:
1. **Parte 1: Tratamento e Higienização dos Dados**
   - Sanitização de caracteres invisíveis e espaços residuais.
   - Deduplicação de registros legados.
   - Identificação e sinalização mandatória de campos com valores nulos (`null` / `NULO`).
2. **Parte 2: Mapeamento da Janela Temporal**
   - Identificação do intervalo entre a primeira e a última data operacional.
   - Estruturação do horizonte temporal para suporte a análises futuras e regras comportamentais.

---
## 1. Tratamento dos Dados e Sinalização de Valores Nulos
Carregamento da base legada e execução dos procedimentos de auditoria e higienização.

In [1]:
import os
import json
import unicodedata
import pandas as pd
from datetime import datetime

caminhos_possiveis = [
    os.path.join("..", "dados", "dados_nivel_1.json"),
    os.path.join("dados", "dados_nivel_1.json")
]
caminho_dados = next((p for p in caminhos_possiveis if os.path.exists(p)), "dados/dados_nivel_1.json")

with open(caminho_dados, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

taxa_cambio = raw_data.get("taxa_cambio_usd_brl", 5.4)
df_raw = pd.DataFrame(raw_data["operacoes"])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio}")
print(f"Total de registros recebidos do legado: {len(df_raw)}")

Taxa de câmbio USD/BRL: 5.4
Total de registros recebidos do legado: 20


In [2]:
def limpar_texto(valor):
    if isinstance(valor, str):
        return "".join(c for c in valor if unicodedata.category(c)[0] != "C").strip()
    return valor

df_limpo = df_raw.map(limpar_texto)
df_tratado = df_limpo.drop_duplicates(subset=["id"], keep="first").copy()
duplicatas_removidas = len(df_raw) - len(df_tratado)

print("=== RELATÓRIO DE HIGIENIZAÇÃO ===")
print(f"• Registros originais: {len(df_raw)}")
print(f"• Duplicidades removidas: {duplicatas_removidas}")
print(f"• Registros válidos pós-deduplicação: {len(df_tratado)}")

=== RELATÓRIO DE HIGIENIZAÇÃO ===
• Registros originais: 20
• Duplicidades removidas: 1
• Registros válidos pós-deduplicação: 19


In [3]:
condicao_nulo = (
    df_tratado.isnull().any(axis=1) | 
    df_tratado.isin(["NULO", "NULL", "None", "nan"]).any(axis=1)
)
registros_nulos = df_tratado[condicao_nulo]

print("=== SINALIZAÇÃO DE REGISTROS NULOS ===")
print(f"Total de operações com inconsistência/nulos: {len(registros_nulos)}\n")

for _, row in registros_nulos.iterrows():
    print("⚠️  ALERTA CRÍTICO DE AUDITORIA (PLD/AML):")
    print(f"   • ID da Operação: {row['id']}")
    print(f"   • Cliente: {row['cliente_id']}")
    print(f"   • Campo com Nulo: 'data' -> {row['data']}")
    print(f"   • Canal / Tipo: {row['canal']} / {row['tipo']}")
    print(f"   • Valor: R$ {row['valor']:,.2f}")
    print(f"   • Observação do Sistema: '{row['observacao']}'")

registros_nulos

=== SINALIZAÇÃO DE REGISTROS NULOS ===
Total de operações com inconsistência/nulos: 1

⚠️  ALERTA CRÍTICO DE AUDITORIA (PLD/AML):
   • ID da Operação: OP-0017
   • Cliente: CLI-A-5
   • Campo com Nulo: 'data' -> None
   • Canal / Tipo: especie / deposito
   • Valor: R$ 4,300.00
   • Observação do Sistema: 'data nao capturada pelo sistema'


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


---
## 2. Análise da Janela Temporal (Espaço entre Datas)

Mapeamento do intervalo entre a primeira e a última data para contextualizar a operação `OP-0017` e estruturar análises cronológicas futuras.

In [4]:
df_temporal = df_tratado[df_tratado["data"].notnull() & (df_tratado["data"] != "")].copy()
df_temporal["data_dt"] = pd.to_datetime(df_temporal["data"], format="%Y-%m-%d")
df_temporal = df_temporal.sort_values(by="data_dt").reset_index(drop=True)

primeira_data = df_temporal["data_dt"].min()
ultima_data = df_temporal["data_dt"].max()
intervalo_dias = (ultima_data - primeira_data).days

op_primeira = df_temporal.iloc[0]
op_ultima = df_temporal.iloc[-1]

print("=== ANÁLISE DO ESPAÇO TEMPORAL ===")
print(f"• Primeira transação registrada: {primeira_data.strftime('%d/%m/%Y')} ({primeira_data.strftime('%d/%m')})")
print(f"  └─ ID: {op_primeira['id']} | Cliente: {op_primeira['cliente_id']} | Valor: R$ {op_primeira['valor']:,.2f}")
print(f"• Última transação registrada:   {ultima_data.strftime('%d/%m/%Y')} ({ultima_data.strftime('%d/%m')})")
print(f"  └─ ID: {op_ultima['id']} | Cliente: {op_ultima['cliente_id']} | Valor: R$ {op_ultima['valor']:,.2f}")
print(f"• Espaço temporal auditado:      {intervalo_dias} dias (de {primeira_data.strftime('%d/%m')} a {ultima_data.strftime('%d/%m')})")

=== ANÁLISE DO ESPAÇO TEMPORAL ===
• Primeira transação registrada: 03/03/2026 (03/03)
  └─ ID: OP-0010 | Cliente: CLI-A-4 | Valor: R$ 3,800.00
• Última transação registrada:   28/03/2026 (28/03)
  └─ ID: OP-0019 | Cliente: CLI-A-6 | Valor: R$ 1,400.00
• Espaço temporal auditado:      25 dias (de 03/03 a 28/03)


In [5]:
resumo_cronologico = df_temporal[['id', 'cliente_id', 'data', 'valor', 'moeda', 'canal', 'tipo', 'contraparte']]
resumo_cronologico.head(10)

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte
0,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA
1,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao
2,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria
3,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria
4,OP-0014,CLI-A-5,2026-03-07,2900,BRL,pix,transferencia_recebida,Delta Transportes
5,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA
6,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME
7,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA
8,OP-0011,CLI-A-4,2026-03-11,5100,BRL,boleto,pagamento,Beta Servicos ME
9,OP-0018,CLI-A-6,2026-03-12,8800,BRL,pix,transferencia_enviada,Alfa Comercio LTDA


---
## 🎯 Conclusões e Próximos Passos de PLD/AML
1. **Base saneada:** 19 operações únicas validadas.
2. **Sinalização Crítica:** A operação `OP-0017` (depósito em espécie de R$ 4.300,00 para `CLI-A-5`) não possui data registrada e exige tratamento de risco regulatório.
3. **Janela Operacional:** Todas as transações com data ocorreram entre **03/03/2026** e **28/03/2026** (25 dias de intervalo), estabelecendo o período para detecção de fracionamento e padrões atípicos.